# Image Clustering (CLIP)

Embeds each survey image with `openai/clip-vit-base-patch32` via the HuggingFace Inference API, then clusters the embeddings with k-means or HDBSCAN. Adds `Image_Cluster#number` and a user-named label column. Shows a UMAP scatter plot for validation.

**Requires:** survey images on NFS storage (`has_images` capability).

A free HuggingFace token improves rate limits. Get one at https://huggingface.co/settings/tokens

In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

SUAVE_TOKEN = ''   # @param {type:"string"}
SUAVE_HOST  = ''   # @param {type:"string"}

_p = su.load_params(token=SUAVE_TOKEN, host=SUAVE_HOST)
if _p is None:
    raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb, or enter token above.')

user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
params        = ''
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'
_prefix       = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
full_notebook_url = _prefix.rstrip('/') + '/lab/tree/SuAVEDispatch.ipynb'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

if not _caps.get('has_images'):
    raise RuntimeError('Full-size images are not available for this survey.')

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Optional HuggingFace token — improves rate limits, not required
# Get a free token at https://huggingface.co/settings/tokens
HF_TOKEN = ''   # @param {type:"string"}

client = su.get_hf_client(HF_TOKEN)

## 1. Load data and configure clustering

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

img_cols = [c for c in df.columns if '#img' in c]
if not img_cols:
    raise ValueError('No #img column found.')
img_col = img_cols[0]

algo_selector = widgets.RadioButtons(
    options=['k-means', 'HDBSCAN'], description='Algorithm:'
)
k_slider = widgets.IntSlider(
    value=6, min=2, max=30, description='k (k-means):',
    continuous_update=False
)
display(algo_selector, k_slider)

## 2. Compute CLIP embeddings

In [ ]:
MODEL = 'openai/clip-vit-base-patch32'
from tqdm.auto import tqdm

embeddings = []
valid_idx  = []

for i, img_name in enumerate(tqdm(df[img_col], desc='Embedding')):
    if not img_name or pd.isna(img_name):
        embeddings.append(None); continue
    img_path = os.path.join(full_images, str(img_name))
    if not os.path.isfile(img_path):
        embeddings.append(None); continue
    try:
        with open(img_path, 'rb') as f:
            img_bytes = f.read()
        emb = client.feature_extraction(img_bytes, model=MODEL)
        embeddings.append(np.array(emb).flatten())
        valid_idx.append(i)
    except Exception:
        embeddings.append(None)

printmd(f"**Embedded {len(valid_idx)}/{len(df)} images.**")

## 3. Cluster and visualize

In [ ]:
from sklearn.cluster import KMeans

X = np.array([e for e in embeddings if e is not None])
algo = algo_selector.value

if algo == 'k-means':
    model  = KMeans(n_clusters=k_slider.value, random_state=42, n_init='auto')
    labels = model.fit_predict(X)
else:
    try:
        import hdbscan
    except ImportError:
        import subprocess, sys as _sys
        subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', 'hdbscan'])
        import hdbscan
    labels = hdbscan.HDBSCAN(min_samples=5).fit_predict(X)

# Map back to full DataFrame index
cluster_col = [None] * len(df)
for j, idx in enumerate(valid_idx):
    cluster_col[idx] = int(labels[j])

df['Image_Cluster#number'] = cluster_col

printmd('**Cluster sizes:**')
display(df['Image_Cluster#number'].value_counts().sort_index().rename('count').to_frame())

# UMAP scatter
try:
    import umap
except ImportError:
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn'])
    import umap

coords = umap.UMAP(n_components=2, random_state=42).fit_transform(X)
unique = sorted(set(labels))
cmap   = plt.cm.get_cmap('tab20', len(unique))

plt.figure(figsize=(8, 6))
for i, lbl in enumerate(unique):
    mask = labels == lbl
    color = 'gray' if lbl == -1 else cmap(i)
    plt.scatter(coords[mask, 0], coords[mask, 1], s=12, alpha=0.6, color=color,
                label=f'Cluster {lbl}' if lbl != -1 else 'Noise')
plt.title('Image clusters (UMAP projection of CLIP embeddings)')
plt.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
input_text = widgets.Text(placeholder='Enter survey name...')
output_text = widgets.Text()

def _bind(sender):
    output_text.value = input_text.value

input_text.on_submit(_bind)
printmd("**Enter survey name, press Enter, then run the next cell:**")
display(input_text, output_text)

In [ ]:
survey_name = output_text.value
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)